<!-- beginner-banner-v2 -->

> 🧭 <strong>비개발자 수강생 안내</strong> — 이 노트북에서 새로 배우는 것: 프롬프트 튜닝과 Few-shot 예시로 <strong>Text-to-SQL 정확도 끌어올리기</strong>.
>
> - 📖 강의 페이지: <a href="https://siapapa.github.io/day2/10-text-to-sql-advanced/" target="_blank" rel="noopener noreferrer">day2/10-text-to-sql-advanced</a>
> - 🆕 처음이라면 → <a href="https://siapapa.github.io/beginners-guide/" target="_blank" rel="noopener noreferrer">비개발자 학습 가이드</a>
> - 🔤 모르는 단어 → <a href="https://siapapa.github.io/appendix/glossary/" target="_blank" rel="noopener noreferrer">용어 사전</a>
> - 🛠️ 환경/접속 막힘 → <a href="https://siapapa.github.io/setup/" target="_blank" rel="noopener noreferrer">사전 준비</a> · <a href="https://siapapa.github.io/appendix/troubleshooting/" target="_blank" rel="noopener noreferrer">트러블슈팅</a>
>
> 외부 링크는 새 탭으로 열리도록 설정돼 있어 Colab 의 리디렉션 경고 페이지를 거치지 않습니다.<br/>
> <strong>셀은 위에서 아래로 차례대로 실행</strong>하세요. 시연용 코드(<code>구경만 하세요</code> 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 08. Text-to-SQL 심화 — 내부 프롬프트 · Few-shot · 테이블 자동 선택
> Day 2 · 10H · 소요 약 50분

## 학습 목표

- `NLSQLTableQueryEngine` 의 **내부 프롬프트**(`get_prompts()`)를 열어 LLM 이 실제로 보는 텍스트를 확인한다.
- 커스텀 `PromptTemplate` 로 **도메인 규칙**과 **Few-shot 예시**를 주입해 SQL 생성 품질을 끌어올린다.
- `ObjectIndex` + `SQLTableRetrieverQueryEngine` 로 **관련 테이블만 자동 선택**하는 RAG 스타일 Text-to-SQL 을 구성한다.
- 같은 질문을 **Before / After** 비교해 프롬프트 튜닝의 효과를 측정한다.

> **선행 조건**
> - `01_postgres_basics.ipynb` 로 병원 DB(`patients`, `doctors`, `visits`, `diagnoses`, `departments`)가 Neon 에 적재되어 있어야 합니다.
> - `03_schema_intelligence.ipynb` 에서 `COMMENT ON` 이 추가된 상태라면 `table_info` 품질이 더 좋아집니다.
> - 이 노트북은 `06_text_to_sql.ipynb` 의 **후속편**입니다. 기본 엔진의 한계 사례가 06 에서 드러났다면, 08 에서 그 한계를 넘어서는 방법을 배웁니다.

In [ ]:
%pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai \
    sqlalchemy psycopg2-binary pandas

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다.
import os

def _load_secret(key: str, required: bool = True) -> None:
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

_load_secret("NEON_DSN", required=True)
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")

## 왜 기본 설정으로는 부족한가?

06 번 노트북에서 경험한 실패 원인을 정리하면 다음과 같습니다.

```
1. 스키마 정보 부족 — COMMENT 가 없거나 빈약하면 LLM 이 컬럼 의미를 오해
2. 프롬프트 품질   — 기본 프롬프트는 한국어 질문에 최적화되지 않음
3. 테이블 선택 오류 — 테이블이 많을 때 관련 없는 테이블까지 프롬프트에 포함
4. 컨텍스트 부족   — 도메인 특수 규칙("completed 만 유효", "최근 = 3개월")을 LLM 이 모름
```

이 노트북에서 쓸 **세 가지 개선 전략**:

| 전략 | 한 줄 설명 | 핵심 API |
|---|---|---|
| ① `table_info` 보강 | 스키마 뒤에 도메인 규칙 텍스트를 붙여 넘긴다 | `PromptTemplate` |
| ② Few-shot 주입 | (질문, SQL) 예시 쌍으로 패턴을 학습시킨다 | `PromptTemplate` |
| ③ 테이블 자동 선택 | 질문과 관련된 테이블만 벡터 검색으로 고른다 | `ObjectIndex`, `SQLTableRetrieverQueryEngine` |

In [ ]:
# ============================================================
# 1. 엔진·SQLDatabase·기본 NLSQLTableQueryEngine 준비
# ============================================================
from sqlalchemy import create_engine
from llama_index.core import SQLDatabase, Settings
from llama_index.core.query_engine import NLSQLTableQueryEngine
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

engine = create_engine(os.environ["NEON_DSN"])

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

HOSPITAL_TABLES = ["patients", "doctors", "visits", "diagnoses", "departments"]

sql_db = SQLDatabase(engine, include_tables=HOSPITAL_TABLES)

# 기본(baseline) 엔진 — 06 번 노트북과 동일 설정
nlq = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=HOSPITAL_TABLES,
)
print("Baseline NLSQLTableQueryEngine ready.")

## 1. 내부 프롬프트 뜯어보기

`query_engine.get_prompts()` 로 LLM 에 전달되는 프롬프트 템플릿을 **직접** 볼 수 있습니다. 이것이 학습의 "아하!" 순간입니다 — LLM 이 실제로 보는 건 바로 이 텍스트입니다.

In [ ]:
# 내부 프롬프트 키 확인
prompts = nlq.get_prompts()

print("사용 중인 프롬프트 키:")
for key in prompts:
    print(f"  - {key}")

In [ ]:
# text_to_sql_prompt 의 템플릿 전문 출력
# ------------------------------------------------------------
# LlamaIndex 0.10+ 부터 NLSQLTableQueryEngine.get_prompts() 가 반환하는
# 키에 prefix 가 붙는 경우가 있다. 예: "sql_retriever:text_to_sql_prompt".
# 따라서 정확 일치 대신 "...text_to_sql_prompt" 로 끝나는 첫 키를 찾는다.
key = next((k for k in prompts if k.endswith("text_to_sql_prompt")), None)
if key is None:
    raise KeyError(
        f"text_to_sql_prompt 키를 찾지 못했습니다. 사용 가능한 키 목록: {list(prompts)}"
    )

print("=" * 60)
print(f"{key} template:")
print("=" * 60)
print(prompts[key].template)


### `{schema}` 변수는 무엇으로 채워지는가?

`{schema}` 자리에는 각 테이블에 대해 `sql_db.get_single_table_info(...)` 가 반환하는 텍스트가 들어갑니다. 여기엔 `CREATE TABLE`, `COMMENT`, 샘플 3행이 포함됩니다. `COMMENT ON` 이 없으면 LLM 은 컬럼 이름만 보고 의미를 추측할 수밖에 없습니다.

In [ ]:
# 각 테이블에 대해 LLM 이 보는 table_info 를 직접 확인
for table in sql_db.get_usable_table_names():
    info = sql_db.get_single_table_info(table)
    print(f"--- {table} ---")
    print(info)
    print()

## 2. 전략 ① — 도메인 지식 주입

비즈니스 규칙(예: `visits.status = 'completed'` 만 유효, "지난달" 의 정의)을 프롬프트의 스키마 블록 뒤에 이어붙여 전달합니다. `text_to_sql_prompt` 를 커스텀 `PromptTemplate` 로 교체하면 됩니다.

In [ ]:
# ============================================================
# 2. 전략 ①: context_info 를 커스텀 프롬프트에 주입
# ============================================================
from llama_index.core.prompts import PromptTemplate

context_info = """
## 도메인 설명
이 데이터베이스는 종합병원의 진료 기록 시스템입니다.

## 주요 비즈니스 규칙
- visits.visit_type: 'outpatient'=외래, 'inpatient'=입원, 'emergency'=응급
- visits.status: 'completed'=완료된 진료만 집계 대상 (cancelled, no_show 제외)
- visits.cost: 원(KRW) 단위, NULL 이면 미청구
- diagnoses.severity: 'mild'=경증, 'moderate'=중등, 'severe'=중증
- patients.gender: 'M'=남성, 'F'=여성
- 나이 계산: EXTRACT(YEAR FROM AGE(birth_date))

## 자주 사용되는 패턴
- "지난달" = visit_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month')
                AND visit_date <  DATE_TRUNC('month', CURRENT_DATE)
- "올해"  = EXTRACT(YEAR FROM visit_date) = EXTRACT(YEAR FROM CURRENT_DATE)
- "재방문" = 같은 patient_id 로 visits 에 2건 이상
"""

enhanced_prompt = PromptTemplate(
    """당신은 PostgreSQL 전문가입니다. 아래 스키마와 도메인 지식을 함께 참고해 SQL 을 작성하세요.

## 데이터베이스 스키마
{schema}
"""
    + context_info
    + """
## 질문
{query_str}

## SQL
"""
)

nlq_enhanced = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=HOSPITAL_TABLES,
    text_to_sql_prompt=enhanced_prompt,
)
print("Enhanced engine (context injection) ready.")

## 3. 전략 ② — 커스텀 프롬프트 + Few-shot

한국어 질문 의도 파악·별칭 규칙·날짜 패턴을 **예시 3개**로 보여 줍니다. LLM 은 예시의 패턴을 쉽게 모방하므로, 도메인 규칙 서술보다 Few-shot 이 더 강력한 경우가 많습니다.

In [ ]:
# ============================================================
# 3. 전략 ②: 커스텀 프롬프트 + Few-shot 예제 3개
# ============================================================
custom_text_to_sql_prompt = PromptTemplate(
    """당신은 PostgreSQL 전문가입니다. 아래 스키마와 규칙을 참고하여 질문에 정확한 SQL 을 작성하세요.

## 데이터베이스 스키마
{schema}

## 도메인 규칙
- visits.status 가 'completed' 인 것만 유효한 진료입니다.
- 비용(cost)이 NULL 이거나 0 인 것은 취소/미방문입니다.
- 나이 계산: EXTRACT(YEAR FROM AGE(birth_date))
- 날짜 필터: PostgreSQL 함수 사용 (DATE_TRUNC, INTERVAL 등)
- 결과는 의미 있는 별칭(AS)을 사용하세요.

## 예시

질문: "전체 환자 수는?"
SQL: SELECT COUNT(*) AS total_patients FROM patients;

질문: "내과 의사 목록을 보여줘"
SQL: SELECT d.name AS doctor_name, d.specialty
     FROM doctors d
     JOIN departments dept ON dept.department_id = d.department_id
     WHERE dept.name = '내과';

질문: "지난달 완료된 진료 건수는?"
SQL: SELECT COUNT(*) AS completed_visits
     FROM visits
     WHERE status = 'completed'
       AND visit_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month')
       AND visit_date <  DATE_TRUNC('month', CURRENT_DATE);

## 지시사항
- SELECT 문만 작성하세요 (DML/DDL 금지).
- SQL 만 반환하세요 (설명 불필요).
- 한국어 질문의 의도를 정확히 파악하세요.

## 질문
{query_str}

## SQL
"""
)

nlq_custom = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=HOSPITAL_TABLES,
    text_to_sql_prompt=custom_text_to_sql_prompt,
)
print("Custom few-shot engine ready.")

## 4. Before / After 비교 실험

동일한 테스트 질문 5개를 **세 엔진**에 모두 흘려 보고 생성된 SQL·성공 여부를 나란히 기록합니다. 단순한 `✅/❌` 로 끝내지 말고, **생성된 SQL 자체**를 비교해야 전략이 어떤 차이를 만드는지 보입니다.

In [ ]:
# ============================================================
# 4. Before / After — 기본 vs context 주입 vs 커스텀+Few-shot
# ============================================================
import pandas as pd

test_questions = [
    "지난달 방문 환자 수는?",
    "진료과별 평균 진료비를 보여주세요.",
    "가장 많이 진단된 질병 Top 5 는?",
    "40세 이상 남성 환자 중 3회 이상 방문한 사람은?",
    "최근에 많이 온 사람은?",  # 모호한 질문
]

def run(engine_, q):
    try:
        r = engine_.query(q)
        return "ok", r.metadata.get("sql_query", "")
    except Exception as e:
        return f"error: {type(e).__name__}", str(e)[:120]

rows = []
for q in test_questions:
    b_status, b_sql = run(nlq, q)
    e_status, e_sql = run(nlq_enhanced, q)
    c_status, c_sql = run(nlq_custom, q)
    rows.append({
        "question": q,
        "basic":  b_status,
        "ctx":    e_status,
        "custom": c_status,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

### 한 질문을 깊게 — 생성된 SQL 나란히 비교

모호하지 않은 질문에서도 `status='completed'` 조건이나 별칭(AS)이 얼마나 달라지는지 한 눈에 보여 줍니다.

In [ ]:
# 한 질문에 대해 세 엔진이 만든 SQL 을 비교
question = "진료과별 평균 진료비를 보여주세요."

print(f"[Q] {question}\n")

for label, eng in [("basic", nlq), ("context", nlq_enhanced), ("custom", nlq_custom)]:
    try:
        r = eng.query(question)
        print(f"--- {label} ---")
        print("SQL:", r.metadata.get("sql_query", "(no sql)"))
        print("A  :", r.response[:200])
        print()
    except Exception as e:
        print(f"--- {label} ---")
        print("ERROR:", type(e).__name__, str(e)[:120])
        print()

## 5. 오답 케이스 — bad query → debug → fix

모호한 질문은 기본 엔진이 **그럴싸하지만 틀린 SQL** 을 만들어 냅니다. 아래 흐름이 "프롬프트 튜닝 루프"의 축소판입니다:

1. 기본 엔진이 만든 SQL 을 읽어 무엇이 빠졌는지(예: `status = 'completed'` 필터) 찾는다.
2. 그 규칙을 커스텀 프롬프트·Few-shot 에 추가한다.
3. 같은 질문을 다시 돌려 수정된 SQL 을 확인한다.

In [ ]:
# ============================================================
# 5. bad query 관찰 → 수정 프롬프트로 재시도
# ============================================================
bad_question = "진료과별 평균 진료비를 보여주세요."

# (1) 기본 엔진: 흔히 status 필터 없이 전체 평균을 냄
r_basic = nlq.query(bad_question)
bad_sql = r_basic.metadata.get("sql_query", "")
print("[BEFORE] basic engine SQL:")
print(bad_sql)
print()

# (2) 관찰한 결함을 규칙으로 명시한 fix 프롬프트
fix_prompt = PromptTemplate(
    """PostgreSQL 전문가입니다.
{schema}

## 필수 규칙
- 진료/비용 관련 집계에는 반드시 visits.status = 'completed' 필터를 WHERE 절에 넣으세요.
- 금액·건수 컬럼은 한글 또는 snake_case 별칭을 달아 주세요.
- JOIN 은 departments 를 통해 진료과 이름을 표시하세요.

## 예시
질문: "진료과별 평균 진료비를 보여주세요."
SQL: SELECT d.name AS department, ROUND(AVG(v.cost), 0) AS avg_cost
     FROM visits v
     JOIN doctors  doc ON doc.doctor_id = v.doctor_id
     JOIN departments d ON d.department_id = doc.department_id
     WHERE v.status = 'completed' AND v.cost IS NOT NULL
     GROUP BY d.name
     ORDER BY avg_cost DESC;

질문: {query_str}
SQL:"""
)

nlq_fixed = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=HOSPITAL_TABLES,
    text_to_sql_prompt=fix_prompt,
)

r_fixed = nlq_fixed.query(bad_question)
print("[AFTER] fixed engine SQL:")
print(r_fixed.metadata.get("sql_query", ""))

## 6. 전략 ③ — 테이블 자동 선택 (`ObjectIndex`)

테이블이 수십 개인 실제 DB 에서는 모든 스키마를 프롬프트에 붙이면 토큰도 터지고 LLM 도 헷갈립니다. 질문과 관련된 **Top-K 개 테이블**만 벡터 검색으로 먼저 고르고, 그 스키마만 프롬프트에 주입하는 RAG 스타일 구성이 `ObjectIndex` 입니다.

| API | 역할 | 한 줄 비유 |
|---|---|---|
| `SQLTableSchema` | 테이블에 자연어 설명을 붙인 객체 | 테이블의 **짧은 소개문** |
| `SQLTableNodeMapping` | 테이블 ↔ 벡터 노드 변환기 | 테이블 이름을 벡터 검색 주소로 바꾸는 어댑터 |
| `ObjectIndex` | 소개문을 임베딩해 "질문 → 관련 테이블 Top-K" 를 반환 | **테이블 전용 검색 엔진** |
| `SQLTableRetrieverQueryEngine` | 테이블 검색 + Text-to-SQL 을 합친 쿼리 엔진 | RAG 형태의 Text-to-SQL |

In [ ]:
# ============================================================
# 6. 전략 ③: SQLTableSchema + ObjectIndex
# ============================================================
# ⚠️ 이 셀은 반드시 위에서부터 끝까지 통째로 실행하세요.
#    `index_cls=` 줄만 수정하고 위쪽 import 를 빠뜨리면 NameError 가 납니다.

# VectorStoreIndex 임포트 — LlamaIndex 0.10.x 의 어느 변형이든 한 번에 처리
try:
    from llama_index.core import VectorStoreIndex
except ImportError:
    try:
        from llama_index.core.indices import VectorStoreIndex  # 일부 빌드
    except ImportError:
        from llama_index.core.indices.vector_store import VectorStoreIndex  # 내부 경로 폴백

from llama_index.core.objects import (
    SQLTableNodeMapping,
    ObjectIndex,
    SQLTableSchema,
)

table_schemas = [
    SQLTableSchema(
        table_name="patients",
        context_str="환자 기본 정보. 이름, 생년월일, 성별, 혈액형 등."
    ),
    SQLTableSchema(
        table_name="doctors",
        context_str="의사 정보. 이름, 소속 진료과, 전공, 급여 등."
    ),
    SQLTableSchema(
        table_name="visits",
        context_str="환자의 진료 방문 기록. 방문일, 진료 유형(외래/입원/응급), 상태, 진료비."
    ),
    SQLTableSchema(
        table_name="diagnoses",
        context_str="진료 시 내려진 진단 기록. ICD-10 코드, 진단명, 중증도."
    ),
    SQLTableSchema(
        table_name="departments",
        context_str="병원 진료과 정보. 진료과명, 위치(층), 전화번호."
    ),
]

table_node_mapping = SQLTableNodeMapping(sql_db)

# index_cls 는 VectorStoreIndex 를 명시. 이전엔 None 이 기본값으로 사용 가능했지만
# 최근 LlamaIndex 에선 None 이 그대로 호출자로 넘어가 TypeError 가 발생한다.
obj_index = ObjectIndex.from_objects(
    table_schemas,
    table_node_mapping,
    index_cls=VectorStoreIndex,
)
print("ObjectIndex built — index_cls:", VectorStoreIndex.__name__)


In [ ]:
# ObjectIndex 기반 SQL Query Engine
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine

nlq_auto = SQLTableRetrieverQueryEngine(
    sql_database=sql_db,
    table_retriever=obj_index.as_retriever(similarity_top_k=3),
    text_to_sql_prompt=custom_text_to_sql_prompt,
)
print("SQLTableRetrieverQueryEngine ready.")

In [ ]:
# ============================================================
# 7. 검증 — "진단" 질문에 diagnoses 테이블이 자동 선택되는지
# ============================================================
response = nlq_auto.query("가장 많이 진단된 질병 Top 5 는?")

print("Answer:", response.response)
print()
print("Generated SQL:")
print(response.metadata.get("sql_query", "(no sql)"))
print()

# LlamaIndex 버전에 따라 metadata 키가 다를 수 있어 안전하게 탐색
picked_tables = response.metadata.get("table_names") or response.metadata.get("tables")
if picked_tables:
    print("Auto-picked tables:", picked_tables)
else:
    # metadata 에 노출이 없으면 SQL 에서 테이블 이름을 추출해 눈으로 확인
    sql = response.metadata.get("sql_query", "").lower()
    candidates = [t for t in HOSPITAL_TABLES if t in sql]
    print("Tables referenced in generated SQL:", candidates)

## 7. 프롬프트 실험 워크시트 — A / B / C

같은 모호한 질문을 세 가지 프롬프트로 시도하며 **프롬프트 엔지니어링이 Text-to-SQL 의 정확도 그 자체**임을 체감합니다.

| 프롬프트 | 특징 |
|---|---|
| A | 기본 — 영문 한 줄 지시, 규칙 없음 |
| B | 모호성 처리 규칙 ("최근" = 최근 3개월, "많이" = 3회 이상 등) |
| C | 규칙 + Few-shot 예시 1개 |

In [ ]:
# ============================================================
# 8. 프롬프트 실험 워크시트
# ============================================================
def experiment(question: str, prompt_template: PromptTemplate, label: str = ""):
    try:
        engine_exp = NLSQLTableQueryEngine(
            sql_database=sql_db,
            tables=HOSPITAL_TABLES,
            text_to_sql_prompt=prompt_template,
        )
        resp = engine_exp.query(question)
        print(f"[{label}] {question}")
        print(f"  SQL: {resp.metadata.get('sql_query','')}")
        print(f"  A  : {resp.response[:160]}")
        return resp
    except Exception as e:
        print(f"[{label}] {question}")
        print(f"  ERROR: {type(e).__name__}: {str(e)[:120]}")
        return None

prompt_a = PromptTemplate(
    "Given the schema:\n{schema}\n\nWrite SQL for: {query_str}\n\nSQL:"
)

prompt_b = PromptTemplate(
    """PostgreSQL 전문가로서 다음 규칙을 따르세요:
{schema}

## 모호성 처리 규칙
- "최근" = 최근 3개월
- "많이" = 3회 이상
- "자주" = 월 평균 2회 이상
- visits.status = 'completed' 만 유효

질문: {query_str}
SQL:"""
)

prompt_c = PromptTemplate(
    """PostgreSQL 전문가입니다.
{schema}

## 규칙
- "최근" = 최근 3개월, "많이" = 3회 이상
- completed 상태만 유효한 진료

## 예시
Q: "최근에 자주 온 환자는?"
SQL: SELECT p.name, COUNT(*) AS visit_count
     FROM visits v JOIN patients p ON p.patient_id = v.patient_id
     WHERE v.status = 'completed'
       AND v.visit_date >= CURRENT_DATE - INTERVAL '3 months'
     GROUP BY p.patient_id, p.name
     HAVING COUNT(*) >= 2
     ORDER BY visit_count DESC;

질문: {query_str}
SQL:"""
)

q = "최근에 많이 온 사람은?"
experiment(q, prompt_a, "A: 기본")
print()
experiment(q, prompt_b, "B: 모호성 규칙")
print()
experiment(q, prompt_c, "C: Few-shot + 규칙")

### 결과 해석 팁

- **A → B**: "최근"·"많이" 해석이 자의적 → 명시된 규칙으로 고정됨.
- **B → C**: 규칙만으로는 JOIN 구성·`HAVING` 사용을 놓치기 쉬움 → Few-shot 1개가 SQL 형태를 견인함.
- Few-shot 예시는 **정답과 형태가 비슷한 1~3개**로 충분합니다. 너무 많이 붙이면 토큰 낭비 + 프롬프트 오염.

## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. Day 1 실패 질문 재시도
**Day 1에서 실패한 질문 3개를 커스텀 프롬프트로 재시도하세요.**

1. Day 1 7H에서 실패했거나 어색했던 질문 3개를 리스트로 정리하세요
2. 각 질문을 `nlq.query(...)` (기본) 와 `nlq_custom.query(...)` (커스텀) 에 동일하게 던지세요
3. 각 시도에서 `response.metadata["sql_query"]` 를 꺼내 두 SQL 을 나란히 출력하세요
4. 단순 ✅/❌ 가 아니라 **생성된 SQL의 차이**(필터 조건, 별칭, JOIN 경로)를 직접 비교 정리하세요

_힌트: 위의 "Before/After 비교 실험" 셀을 참고해 try/except 로 두 엔진을 모두 호출하면 됩니다. 정답 코드는 숨겨져 있으니, 실패 질문을 본인 메모에서 꺼내 직접 작성해 보세요._


In [ ]:
# ============================================================
# 실습 과제 — Day 1 실패 질문 재시도
# ============================================================

# 실습 1: Day 1 실패 질문 재시도
# TODO: Day 1 7H에서 실패한 질문 3개를 nlq / nlq_custom 양쪽에 던지고
#       response.metadata["sql_query"] 를 나란히 비교하세요.
# 여기에 구현하세요.


## 10H 핵심 정리

- `get_prompts()` 로 LLM 이 실제로 보는 **내부 프롬프트**를 직접 확인할 수 있다.
- 세 가지 튜닝 레버: **도메인 지식 주입 / Few-shot / 테이블 자동 선택(ObjectIndex)**.
- "bad query → debug → fix" 루프는 프롬프트 엔지니어링의 기본 감각이자, Day 3 Vanna 자가학습의 예고편입니다 — "수동으로 하던 것을 Vanna 가 자동화" 합니다.

## 다음 노트북에서는…

다음 **`09_gradio_chatbot.ipynb`** (Day 2 · 11~12H) 에서 지금까지 만든 Text-to-SQL 엔진을 **멀티턴 상담사**로 확장합니다. `ChatState` 로 대화 맥락을 누적하고, `SQLGuardrail` 로 DELETE/DROP 을 차단한 뒤, **Gradio `ChatInterface`** 로 `share=True` 공개 URL 챗봇까지 띄워 봅니다.